In [ ]:
print("Installing libraries. Any 'restart kernel' messages that follow are handled automatically. No action needed.")

In [ ]:
%pip install -qq backoff==2.2.1 databricks-openai==0.15.0 databricks-agents==1.10.2 mlflow-skinny[databricks]==3.12.0 \
    databricks-langchain[memory]==0.19.0 langchain-community==0.4.1 langchain==1.2.10 \
    unitycatalog-langchain[databricks]==0.4.0 unitycatalog-ai[databricks]==0.4.0 \
    databricks-sdk==0.94.0 langgraph-checkpoint-postgres==3.0.1 reportlab==4.5.1 uv==0.11.19 pyyaml==6.0.3 \
    langgraph==1.0.10 langgraph-checkpoint==3.0.1 langgraph-prebuilt==1.0.8 \
    databricks-vectorsearch==0.68 \
    typing_extensions>=4.10
dbutils.library.restartPython()

In [ ]:
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

In [ ]:
includes_dir = "./Includes"
course_dir = os.path.dirname(includes_dir)
sys.path.insert(0, course_dir)

config_path = f"{includes_dir}/config/config_lakebase_memory.yaml"

In [ ]:
from Includes import setup_demo_environment
import tempfile, io, contextlib

# Use empty temp dirs so the setup does NOT register demo agents or copy artifacts
_empty_agents_dir = tempfile.mkdtemp(prefix="capstone_empty_agents_")
_empty_configs_dir = tempfile.mkdtemp(prefix="capstone_empty_configs_")
_empty_tools_dir = tempfile.mkdtemp(prefix="capstone_empty_tools_")

# Silence the shared orchestrator's verbose step-by-step logging; the single
# Configuration Values table at the end of this setup is the learner-facing
# summary. stderr is left alone so genuine errors still surface.
with contextlib.redirect_stdout(io.StringIO()):
    env = setup_demo_environment(
        config_path=config_path,
        agents_dir=_empty_agents_dir,
        agent_configs_dir=_empty_configs_dir,
        agent_tools_dir=_empty_tools_dir,
    )

catalog_name = env["catalog_name"]

# Create a dedicated 'bakehouse' schema for the capstone lab and set as default
schema_name = "bakehouse"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}");
spark.sql(f"USE CATALOG {catalog_name}");
spark.sql(f"USE SCHEMA {schema_name}");

In [ ]:
# Experiment location for MLflow tracing
user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_location = f"/Workspace/Users/{user_email}/bakehouse_capstone_experiment"

In [ ]:
# ---- Derive the Lakebase project name (created manually during Section B4) ----
local_part = user_email.lower().split("@")[0]
cleaned = local_part.replace(".", "-").replace("_", "-")
lakebase_project_name = f"bakehouse-{cleaned}"
# Databricks Apps require app names <= 30 chars. Truncate the cleaned username
# to its last 19 chars so the prefix + truncated tail stays under the limit.
app_name = f"bakehouse-{cleaned[-19:]}"

def verify_lakebase_and_patch_bundle(project_name, app_dir):
    """Verify Lakebase project exists and add the postgres resource to databricks.yml."""
    from databricks.sdk import WorkspaceClient
    import yaml, os

    w = WorkspaceClient()
    api_path = f"/api/2.0/postgres/projects/{project_name}/branches/production/databases"
    resp = w.api_client.do("GET", api_path)
    dbs = resp.get("databases", [])
    db = next((d for d in dbs if d.get("status", {}).get("postgres_database") == "databricks_postgres"), None)
    if db is None:
        raise RuntimeError(
            f"Lakebase project '{project_name}' has no databricks_postgres database. "
            "Please ensure you completed Section B2a."
        )
    db_id = db["status"]["database_id"]

    yml_path = os.path.join(app_dir, "databricks.yml")

    with open(yml_path, "r") as f:
        bundle = yaml.safe_load(f)

    app_key = list(bundle["resources"]["apps"].keys())[0]
    resources = [r for r in bundle["resources"]["apps"][app_key].get("resources", [])
                 if r.get("name") not in ("database", "postgres", "lakebase")]
    resources.append({
        "name": "database",
        "postgres": {
            "branch": f"projects/{project_name}/branches/production",
            "database": f"projects/{project_name}/branches/production/databases/{db_id}",
            "permission": "CAN_CONNECT_AND_CREATE"
        }
    })
    bundle["resources"]["apps"][app_key]["resources"] = resources

    with open(yml_path, "w") as f:
        yaml.dump(bundle, f, default_flow_style=False, sort_keys=False)

    print(f"Lakebase project verified: {project_name}")
    print(f"Database ID:               {db_id}")
    print(f"databricks.yml updated with Lakebase database resource.")
    return db_id

In [ ]:
# ---- [4] Create bakehouse-agent-app folder with all application files ----
# Resolve notebook directory (works on Databricks repos and workspace notebooks)
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_workspace_prefix = "/Workspace"
if _nb_path.startswith(_workspace_prefix):
    _nb_abs = _nb_path
else:
    _nb_abs = _workspace_prefix + _nb_path
_nb_dir = os.path.dirname(_nb_abs)
_app_dir = os.path.join(_nb_dir, "bakehouse-agent-app")
os.makedirs(_app_dir, exist_ok=True)

# ---- agent.py ----
_agent_py = r'''import os, json, uuid, asyncio, logging
import mlflow
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, InjectedStore
from langgraph.store.base import BaseStore
from databricks_langchain import ChatDatabricks, AsyncCheckpointSaver, AsyncDatabricksStore
from typing import Annotated

logger = logging.getLogger(__name__)

# --- Config from environment ---
_endpoint = os.environ.get("SERVING_ENDPOINT_NAME", "databricks-claude-sonnet-4-6")
_catalog = os.environ.get("CATALOG_NAME", "")
_schema = os.environ.get("SCHEMA_NAME", "bakehouse")
_lb_project = os.environ.get("LAKEBASE_AUTOSCALING_PROJECT", "")
_lb_branch = os.environ.get("LAKEBASE_AUTOSCALING_BRANCH", "production")
_experiment_id = os.environ.get("MLFLOW_EXPERIMENT_ID", "")
_func1 = os.environ.get("FUNCTION_1_NAME", "top_products_by_franchise")
_func2 = os.environ.get("FUNCTION_2_NAME", "franchise_performance")
_embedding_endpoint = os.environ.get("EMBEDDING_ENDPOINT", "databricks-gte-large-en")
_embedding_dims = int(os.environ.get("EMBEDDING_DIMS", "1024"))

SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", """You are the Bakehouse Franchise Sales Assistant. You help franchise managers analyze their sales data, compare performance across locations, and track product trends.

At the START of every conversation:
1. Call recall_memories to check if you already know this manager
2. If memories exist, greet them by name and reference their franchise

When you learn new facts about a manager (name, franchise, preferred products, reporting preferences), call save_memory to store them.

Bakehouse operates 48 franchises across the US, Japan, Australia, Canada, and Europe.
Products include: Austin Almond Biscotti, Golden Gate Ginger, Tokyo Tidbits, Outback Oatmeal, Pearly Pies, and Orchard Oasis.
Always be specific with numbers and provide actionable insights.""")

# MLflow setup
mlflow.set_tracking_uri("databricks")
mlflow.langchain.autolog()
if _experiment_id:
    mlflow.set_experiment(experiment_id=_experiment_id)

# --- Memory tools ---
@tool
def save_memory(key: str, value: str, config: RunnableConfig, store: Annotated[BaseStore, InjectedStore]) -> str:
    """Save a fact about this franchise manager (name, franchise, preferred product, etc.)."""
    user_id = config["configurable"].get("user_id", "anonymous")
    store.put(("managers", user_id), key, {"content": value})
    return f"Saved '{key}' = '{value}'"

@tool
def recall_memories(query: str, config: RunnableConfig, store: Annotated[BaseStore, InjectedStore]) -> str:
    """Search this manager's stored facts by semantic similarity."""
    user_id = config["configurable"].get("user_id", "anonymous")
    results = store.search(("managers", user_id), query=query, limit=5)
    if not results:
        return "No memories found for this manager."
    return "Recalled:\n" + "\n".join(f"  [{r.key}]: {r.value['content']}" for r in results)

# --- MCP tool discovery ---
_mcp_tools = None

async def _get_mcp_tools():
    global _mcp_tools
    if _mcp_tools is not None:
        return _mcp_tools
    _mcp_tools = []
    if _catalog and _schema and _func1:
        try:
            from unitycatalog.ai.langchain_toolkit import UCFunctionToolkit
            names = [f"{_catalog}.{_schema}.{_func1}"]
            if _func2:
                names.append(f"{_catalog}.{_schema}.{_func2}")
            toolkit = UCFunctionToolkit(function_names=names)
            _mcp_tools = toolkit.tools
            for t in _mcp_tools:
                t.name = t.name.replace(".", "_")
        except Exception as e:
            logger.warning(f"UC tools not loaded: {e}")
    return _mcp_tools

# --- Graph builder ---
def _build_graph(checkpointer, store, model_with_tools, tools):
    def agent_node(state: MessagesState):
        msgs = [SystemMessage(content=SYSTEM_PROMPT)] + list(state["messages"])
        return {"messages": [model_with_tools.invoke(msgs)]}

    def should_continue(state: MessagesState):
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return END

    wf = StateGraph(MessagesState)
    wf.add_node("agent", agent_node)
    wf.add_node("tools", ToolNode(tools))
    wf.add_edge(START, "agent")
    wf.add_conditional_edges("agent", should_continue, ["tools", END])
    wf.add_edge("tools", "agent")
    return wf.compile(checkpointer=checkpointer, store=store)

# --- Async execution ---
async def _run_async(messages, thread_id, user_id, model_with_tools, tools):
    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    lk = {"project": _lb_project, "branch": _lb_branch}

    async with AsyncDatabricksStore(**lk, embedding_endpoint=_embedding_endpoint, embedding_dims=_embedding_dims) as store:
        await store.setup()
        async with AsyncCheckpointSaver(**lk) as cp:
            graph = _build_graph(cp, store, model_with_tools, tools)
            result = await graph.ainvoke({"messages": messages}, config)

    return result["messages"][-1].content

# --- Public interface ---
@mlflow.trace(name="bakehouse_agent_run", span_type="AGENT")
async def run_agent(messages, thread_id=None, user_id="anonymous"):
    thread_id = thread_id or str(uuid.uuid4())
    mcp_tools = await _get_mcp_tools()
    all_tools = [save_memory, recall_memories] + mcp_tools

    model = ChatDatabricks(endpoint=_endpoint)
    model_with_tools = model.bind_tools(all_tools)

    output = await _run_async(messages, thread_id, user_id, model_with_tools, all_tools)
    return {"output": output, "thread_id": thread_id}
'''

with open(os.path.join(_app_dir, "agent.py"), "w") as f:
    f.write(_agent_py)

# ---- server.py ----
_server_py = '''import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from typing import Optional
from pathlib import Path
from agent import run_agent

app = FastAPI()

class ChatRequest(BaseModel):
    input: list
    thread_id: Optional[str] = None

@app.get("/")
async def root():
    return HTMLResponse(Path("chat.html").read_text())

@app.post("/invocations")
async def invoke(req: ChatRequest, request: Request):
    raw_user = request.headers.get("X-Forwarded-Preferred-Username", "anonymous")
    user_id = raw_user.replace(".", "_").replace("@", "_at_")
    messages = [{"role": m.get("role", "user"), "content": m.get("content", "")} for m in req.input]
    result = await run_agent(messages, thread_id=req.thread_id, user_id=user_id)
    return result

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open(os.path.join(_app_dir, "server.py"), "w") as f:
    f.write(_server_py)

# ---- chat.html ----
_chat_html = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Bakehouse Sales Assistant</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700&family=Inter:wght@400;500;600&display=swap');

  * { margin: 0; padding: 0; box-sizing: border-box; }

  body {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    background: linear-gradient(135deg, #FFF8DC 0%, #FAEBD7 50%, #FFE4C4 100%);
    height: 100vh;
    display: flex;
    flex-direction: column;
    color: #3E2723;
  }

  /* Header */
  .header {
    background: linear-gradient(135deg, #8B4513 0%, #A0522D 50%, #D2691E 100%);
    padding: 16px 24px;
    display: flex;
    align-items: center;
    gap: 14px;
    box-shadow: 0 4px 20px rgba(139, 69, 19, 0.3);
    position: relative;
    overflow: hidden;
  }
  .header::before {
    content: '';
    position: absolute;
    top: -50%;
    left: -50%;
    width: 200%;
    height: 200%;
    background: repeating-linear-gradient(
      45deg,
      transparent,
      transparent 30px,
      rgba(255,255,255,0.03) 30px,
      rgba(255,255,255,0.03) 60px
    );
  }
  .header-icon {
    font-size: 36px;
    z-index: 1;
    filter: drop-shadow(0 2px 4px rgba(0,0,0,0.2));
  }
  .header-text { z-index: 1; }
  .header-title {
    font-family: 'Playfair Display', Georgia, serif;
    font-size: 22px;
    font-weight: 700;
    color: #FFF8DC;
    letter-spacing: 0.5px;
  }
  .header-subtitle {
    font-size: 12px;
    color: #FFDEAD;
    margin-top: 2px;
    font-weight: 500;
    letter-spacing: 0.3px;
  }

  /* Chat area */
  .chat-container {
    flex: 1;
    overflow-y: auto;
    padding: 24px;
    display: flex;
    flex-direction: column;
    gap: 16px;
    scroll-behavior: smooth;
  }
  .chat-container::-webkit-scrollbar { width: 6px; }
  .chat-container::-webkit-scrollbar-track { background: transparent; }
  .chat-container::-webkit-scrollbar-thumb {
    background: #D2B48C;
    border-radius: 3px;
  }

  /* Welcome card */
  .welcome-card {
    background: linear-gradient(135deg, #FFFAF0 0%, #FFF8DC 100%);
    border: 2px solid #DEB887;
    border-radius: 16px;
    padding: 28px;
    text-align: center;
    margin: 20px auto;
    max-width: 520px;
    box-shadow: 0 4px 16px rgba(139, 69, 19, 0.1);
  }
  .welcome-card h2 {
    font-family: 'Playfair Display', Georgia, serif;
    color: #8B4513;
    font-size: 20px;
    margin-bottom: 12px;
  }
  .welcome-card p {
    color: #6D4C41;
    font-size: 14px;
    line-height: 1.6;
    margin-bottom: 16px;
  }
  .welcome-suggestions {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    justify-content: center;
  }
  .suggestion-chip {
    background: #FFF;
    border: 1.5px solid #D2691E;
    border-radius: 20px;
    padding: 8px 16px;
    font-size: 13px;
    color: #8B4513;
    cursor: pointer;
    transition: all 0.2s ease;
    font-weight: 500;
  }
  .suggestion-chip:hover {
    background: #D2691E;
    color: #FFF;
    transform: translateY(-1px);
    box-shadow: 0 3px 8px rgba(210, 105, 30, 0.3);
  }

  /* Messages */
  .message-row {
    display: flex;
    gap: 12px;
    max-width: 780px;
    width: 100%;
    margin: 0 auto;
    animation: fadeSlideIn 0.3s ease;
  }
  @keyframes fadeSlideIn {
    from { opacity: 0; transform: translateY(8px); }
    to { opacity: 1; transform: translateY(0); }
  }

  .message-row.user { flex-direction: row-reverse; }
  .avatar {
    width: 38px;
    height: 38px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 20px;
    flex-shrink: 0;
    margin-top: 4px;
  }
  .avatar.agent {
    background: linear-gradient(135deg, #FAEBD7, #DEB887);
    border: 2px solid #D2691E;
    box-shadow: 0 2px 8px rgba(210, 105, 30, 0.2);
  }
  .avatar.user {
    background: linear-gradient(135deg, #FF6B35, #FF8C42);
    border: 2px solid #E55A2B;
    color: white;
    font-size: 16px;
    font-weight: 600;
  }

  .bubble {
    padding: 14px 18px;
    border-radius: 16px;
    font-size: 14px;
    line-height: 1.65;
    max-width: 600px;
    word-wrap: break-word;
    white-space: pre-wrap;
  }
  .bubble.agent {
    background: #FFF8DC;
    border: 1.5px solid #DEB887;
    border-radius: 16px 16px 16px 4px;
    color: #3E2723;
    box-shadow: 0 2px 8px rgba(139, 69, 19, 0.08);
  }
  .bubble.agent strong { color: #8B4513; }
  .bubble.agent ul, .bubble.agent ol { margin: 6px 0 6px 18px; padding: 0; }
  .bubble.agent li { margin: 3px 0; }
  .bubble.agent code { font-size: 13px; }
  .bubble.user {
    background: linear-gradient(135deg, #FF6B35, #FF7F42);
    border-radius: 16px 16px 4px 16px;
    color: #FFFFFF;
    box-shadow: 0 2px 10px rgba(255, 107, 53, 0.25);
  }

  /* Thinking indicator */
  .thinking-row {
    display: flex;
    gap: 12px;
    max-width: 780px;
    width: 100%;
    margin: 0 auto;
    animation: fadeSlideIn 0.3s ease;
  }
  .thinking-bubble {
    background: #FFF8DC;
    border: 1.5px solid #DEB887;
    border-radius: 16px 16px 16px 4px;
    padding: 14px 18px;
    display: flex;
    align-items: center;
    gap: 10px;
    color: #8B4513;
    font-size: 14px;
    font-style: italic;
  }
  .thinking-dots span {
    display: inline-block;
    width: 8px;
    height: 8px;
    background: #D2691E;
    border-radius: 50%;
    animation: bounce 1.4s ease-in-out infinite;
    margin-right: 4px;
  }
  .thinking-dots span:nth-child(2) { animation-delay: 0.2s; }
  .thinking-dots span:nth-child(3) { animation-delay: 0.4s; }
  @keyframes bounce {
    0%, 80%, 100% { transform: translateY(0); opacity: 0.4; }
    40% { transform: translateY(-8px); opacity: 1; }
  }

  /* Input area */
  .input-area {
    background: #FFFAF0;
    border-top: 2px solid #DEB887;
    padding: 16px 24px;
    display: flex;
    gap: 12px;
    align-items: flex-end;
    max-width: 100%;
  }
  .input-wrapper {
    flex: 1;
    max-width: 780px;
    margin: 0 auto;
    display: flex;
    gap: 12px;
    align-items: flex-end;
    width: 100%;
  }
  #user-input {
    flex: 1;
    padding: 12px 18px;
    border: 2px solid #DEB887;
    border-radius: 24px;
    font-size: 14px;
    font-family: 'Inter', sans-serif;
    background: #FFFFFF;
    color: #3E2723;
    outline: none;
    transition: border-color 0.2s, box-shadow 0.2s;
    resize: none;
    max-height: 120px;
    min-height: 46px;
    line-height: 1.4;
  }
  #user-input:focus {
    border-color: #D2691E;
    box-shadow: 0 0 0 3px rgba(210, 105, 30, 0.15);
  }
  #user-input::placeholder { color: #BC8F6F; }

  #send-btn {
    width: 46px;
    height: 46px;
    border-radius: 50%;
    border: none;
    background: linear-gradient(135deg, #D2691E, #FF6B35);
    color: white;
    font-size: 20px;
    cursor: pointer;
    display: flex;
    align-items: center;
    justify-content: center;
    transition: all 0.2s ease;
    flex-shrink: 0;
    box-shadow: 0 2px 8px rgba(210, 105, 30, 0.3);
  }
  #send-btn:hover {
    transform: scale(1.05);
    box-shadow: 0 4px 12px rgba(210, 105, 30, 0.4);
  }
  #send-btn:disabled {
    background: #C9B99A;
    cursor: not-allowed;
    transform: none;
    box-shadow: none;
  }

  /* Footer */
  .footer {
    text-align: center;
    padding: 8px;
    font-size: 11px;
    color: #A0876E;
    background: #FFF5E6;
    border-top: 1px solid #EDE0D0;
  }
</style>
</head>
<body>

<div class="header">
  <div class="header-icon">&#x1F9C1;</div>
  <div class="header-text">
    <div class="header-title">Bakehouse Sales Assistant</div>
    <div class="header-subtitle">Franchise Analytics &bull; Powered by AI</div>
  </div>
</div>

<div class="chat-container" id="chat-container">
  <div class="welcome-card">
    <h2>&#x1F36A; Welcome to Bakehouse HQ</h2>
    <p>I'm your franchise sales assistant. Ask me about product trends, franchise performance, revenue comparisons, and more across all 48 locations.</p>
    <div class="welcome-suggestions">
      <div class="suggestion-chip" onclick="sendSuggestion(this)">Top products in Tokyo?</div>
      <div class="suggestion-chip" onclick="sendSuggestion(this)">Compare US vs. Europe revenue</div>
      <div class="suggestion-chip" onclick="sendSuggestion(this)">Best performing franchise?</div>
    </div>
  </div>
</div>

<div class="input-area">
  <div class="input-wrapper">
    <textarea id="user-input" rows="1" placeholder="Ask about your franchise sales..." onkeydown="handleKey(event)" oninput="autoResize(this)"></textarea>
    <button id="send-btn" onclick="sendMessage()">&#x27A4;</button>
  </div>
</div>

<div class="footer">Powered by Databricks Lakebase + LangGraph</div>

<script>
  let threadId = null;
  const chatContainer = document.getElementById('chat-container');
  const userInput = document.getElementById('user-input');
  const sendBtn = document.getElementById('send-btn');

  function autoResize(el) {
    el.style.height = 'auto';
    el.style.height = Math.min(el.scrollHeight, 120) + 'px';
  }

  function handleKey(e) {
    if (e.key === 'Enter' && !e.shiftKey) {
      e.preventDefault();
      sendMessage();
    }
  }

  function sendSuggestion(el) {
    userInput.value = el.textContent;
    sendMessage();
  }

  function renderMarkdown(text) {
    // Lightweight markdown to HTML converter
    let html = text
      .replace(/&/g, '&amp;').replace(/</g, '&lt;').replace(/>/g, '&gt;')  // escape HTML
      .replace(/\*\*(.+?)\*\*/g, '<strong>$1</strong>')                     // **bold**
      .replace(/\*(.+?)\*/g, '<em>$1</em>')                                 // *italic*
      .replace(/`(.+?)`/g, '<code style="background:#F5E6D3;padding:1px 5px;border-radius:3px;font-size:13px;">$1</code>')  // `code`
      .replace(/^### (.+)$/gm, '<h4 style="margin:12px 0 6px;color:#8B4513;">$1</h4>')
      .replace(/^## (.+)$/gm, '<h3 style="margin:14px 0 8px;color:#8B4513;">$1</h3>')
      .replace(/^# (.+)$/gm, '<h2 style="margin:16px 0 8px;color:#8B4513;">$1</h2>')
      .replace(/^- (.+)$/gm, '<li style="margin:2px 0;">$1</li>')           // - bullets
      .replace(/^(\d+)\. (.+)$/gm, '<li style="margin:2px 0;">$2</li>')     // 1. numbered
      .replace(/(<li.*<\/li>\n?)+/g, '<ul style="margin:8px 0 8px 16px;padding:0;">$&</ul>')  // wrap lists
      .replace(/\n\n/g, '</p><p style="margin:8px 0;">')                     // paragraphs
      .replace(/\n/g, '<br>');                                               // line breaks
    return '<p style="margin:0;">' + html + '</p>';
  }

  function addMessage(role, text) {
    // Remove welcome card on first message
    const welcome = chatContainer.querySelector('.welcome-card');
    if (welcome) welcome.remove();

    const row = document.createElement('div');
    row.className = 'message-row ' + role;

    const avatar = document.createElement('div');
    avatar.className = 'avatar ' + role;
    avatar.innerHTML = role === 'agent' ? '&#x1F35E;' : '&#x1F464;';

    const bubble = document.createElement('div');
    bubble.className = 'bubble ' + role;
    if (role === 'agent') {
      bubble.innerHTML = renderMarkdown(text);
    } else {
      bubble.textContent = text;
    }

    row.appendChild(avatar);
    row.appendChild(bubble);
    chatContainer.appendChild(row);
    chatContainer.scrollTop = chatContainer.scrollHeight;
  }

  function showThinking() {
    const row = document.createElement('div');
    row.className = 'thinking-row';
    row.id = 'thinking';

    const avatar = document.createElement('div');
    avatar.className = 'avatar agent';
    avatar.innerHTML = '&#x1F35E;';

    const bubble = document.createElement('div');
    bubble.className = 'thinking-bubble';
    bubble.innerHTML = '<div class="thinking-dots"><span></span><span></span><span></span></div> Baking up a response...';

    row.appendChild(avatar);
    row.appendChild(bubble);
    chatContainer.appendChild(row);
    chatContainer.scrollTop = chatContainer.scrollHeight;
  }

  function hideThinking() {
    const el = document.getElementById('thinking');
    if (el) el.remove();
  }

  async function sendMessage() {
    const text = userInput.value.trim();
    if (!text) return;

    userInput.value = '';
    userInput.style.height = 'auto';
    sendBtn.disabled = true;
    addMessage('user', text);
    showThinking();

    try {
      const payload = {
        input: [{ role: 'user', content: text }]
      };
      if (threadId) payload.thread_id = threadId;

      const resp = await fetch('/invocations', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(payload)
      });
      const data = await resp.json();
      threadId = data.thread_id || threadId;
      hideThinking();
      addMessage('agent', data.output || 'No response received.');
    } catch (err) {
      hideThinking();
      addMessage('agent', 'Something went wrong. Please try again.');
      console.error(err);
    }

    sendBtn.disabled = false;
    userInput.focus();
  }
</script>
</body>
</html>
'''

with open(os.path.join(_app_dir, "chat.html"), "w") as f:
    f.write(_chat_html)

# ---- app.yaml ----
_app_yaml = f"""env:
  - name: SERVING_ENDPOINT_NAME
    valueFrom: serving-endpoint
  - name: MLFLOW_EXPERIMENT_ID
    valueFrom: experiment
  - name: CATALOG_NAME
    value: "{catalog_name}"
  - name: SCHEMA_NAME
    value: "bakehouse"
  - name: LAKEBASE_AUTOSCALING_PROJECT
    value: "{lakebase_project_name}"
  - name: LAKEBASE_AUTOSCALING_BRANCH
    value: "production"
  - name: FUNCTION_1_NAME
    value: "top_products_by_franchise"
  - name: FUNCTION_2_NAME
    value: "franchise_performance"
  - name: EMBEDDING_ENDPOINT
    value: "databricks-gte-large-en"
  - name: EMBEDDING_DIMS
    value: "1024"

command: ["python", "server.py"]
"""

with open(os.path.join(_app_dir, "app.yaml"), "w") as f:
    f.write(_app_yaml)

# ---- requirements.txt ----
_requirements = """databricks-langchain[memory]==0.19.0
unitycatalog-langchain[databricks]==0.4.0
unitycatalog-ai[databricks]==0.4.0
langchain==1.2.10
langchain-community==0.4.1
fastapi==0.136.3
uvicorn==0.49.0
python-dotenv==1.2.2
langgraph-checkpoint-postgres==3.0.1
langgraph-prebuilt==1.0.8
databricks-vectorsearch==0.68
"""

with open(os.path.join(_app_dir, "requirements.txt"), "w") as f:
    f.write(_requirements)

# ---- databricks.yml ----
_databricks_yml = f"""bundle:
  name: {app_name}

variables:
  catalog_name:
    default: {catalog_name}
  lakebase_autoscaling_project:
    default: {lakebase_project_name}
  function_1_name:
    default: top_products_by_franchise
  function_2_name:
    default: franchise_performance
  embedding_endpoint:
    default: databricks-gte-large-en
  embedding_dims:
    default: "1024"

resources:
  experiments:
    bakehouse_experiment:
      name: /Users/${{workspace.current_user.userName}}/bakehouse_app_experiment

  apps:
    {app_name}:
      name: {app_name}
      description: "Bakehouse Franchise Sales Assistant with stateful memory"
      source_code_path: .
      config:
        command:
          - python
          - server.py
        env:
          - name: SERVING_ENDPOINT_NAME
            value: databricks-claude-sonnet-4-6
          - name: MLFLOW_EXPERIMENT_ID
            value: ${{resources.experiments.bakehouse_experiment.id}}
          - name: CATALOG_NAME
            value: ${{var.catalog_name}}
          - name: SCHEMA_NAME
            value: bakehouse
          - name: LAKEBASE_AUTOSCALING_PROJECT
            value: ${{var.lakebase_autoscaling_project}}
          - name: LAKEBASE_AUTOSCALING_BRANCH
            value: production
          - name: FUNCTION_1_NAME
            value: ${{var.function_1_name}}
          - name: FUNCTION_2_NAME
            value: ${{var.function_2_name}}
          - name: EMBEDDING_ENDPOINT
            value: ${{var.embedding_endpoint}}
          - name: EMBEDDING_DIMS
            value: ${{var.embedding_dims}}
      resources:
        - name: serving-endpoint
          serving_endpoint:
            name: databricks-claude-sonnet-4-6
            permission: CAN_QUERY
        - name: experiment
          experiment:
            experiment_id: ${{resources.experiments.bakehouse_experiment.id}}
            permission: CAN_EDIT
        - name: uc-function-1
          uc_securable:
            securable_full_name: ${{var.catalog_name}}.bakehouse.${{var.function_1_name}}
            securable_type: FUNCTION
            permission: EXECUTE
        - name: uc-function-2
          uc_securable:
            securable_full_name: ${{var.catalog_name}}.bakehouse.${{var.function_2_name}}
            securable_type: FUNCTION
            permission: EXECUTE

targets:
  dev:
    default: true
"""

with open(os.path.join(_app_dir, "databricks.yml"), "w") as f:
    f.write(_databricks_yml)

# ---- pyproject.toml ----
_pyproject_toml = """[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "bakehouse-agent-app"
version = "0.1.0"
description = "Bakehouse Franchise Sales Assistant - Stateful AI Agent with Lakebase Memory"
requires-python = ">=3.11"
dependencies = [
    "databricks-langchain[memory]==0.19.0",
    "databricks-sdk==0.94.0",
    "fastapi==0.136.3",
    "uvicorn==0.49.0",
    "python-dotenv==1.2.2",
    "unitycatalog-langchain[databricks]==0.4.0",
    "unitycatalog-ai[databricks]==0.4.0",
    "langchain==1.2.10",
    "langchain-community==0.4.1",
    "langgraph==1.0.10",
    "langgraph-checkpoint==3.0.1",
    "langgraph-checkpoint-postgres==3.0.1",
    "langgraph-prebuilt==1.0.8",
    "typing_extensions>=4.10",
]

[project.scripts]
start-server = "server:app"
"""

with open(os.path.join(_app_dir, "pyproject.toml"), "w") as f:
    f.write(_pyproject_toml)

In [ ]:
# Write the agent config YAML with resolved values
config_file_path = "./artifacts/configs/bakehouse_agent_config.yaml"
os.makedirs(os.path.dirname(config_file_path), exist_ok=True)

config_content = f"""SYSTEM_PROMPT: |
  You are the Bakehouse Franchise Sales Assistant. You help franchise managers analyze their sales data, compare performance across locations, and track product trends.

  At the START of every conversation:
  1. Call recall_memories to check if you already know this manager
  2. If memories exist, greet them by name and reference their franchise

  When you learn new facts about a manager (name, franchise, preferred products, reporting preferences), call save_memory to store them.

  You have access to these tools:
  - top_products_by_franchise: Find best-selling products by franchise or country
  - franchise_performance: Compare revenue and orders across franchise locations
  - save_memory: Store facts about this manager for future sessions
  - recall_memories: Retrieve previously stored facts about this manager

  Bakehouse operates 48 franchises across the US, Japan, Australia, Canada, and Europe.
  Products include: Austin Almond Biscotti, Golden Gate Ginger, Tokyo Tidbits, Outback Oatmeal, Pearly Pies, and Orchard Oasis.
  All prices are in USD. The dataset covers transactions from 2024.

  Always be specific with numbers and provide actionable insights for franchise managers.

LLM_ENDPOINT_NAME: "databricks-claude-sonnet-4-6"
CATALOG_NAME: "{catalog_name}"
SCHEMA_NAME: "bakehouse"
LAKEBASE_AUTOSCALING_PROJECT: "{lakebase_project_name}"
LAKEBASE_AUTOSCALING_BRANCH: "production"
TOOL1: "top_products_by_franchise"
TOOL2: "franchise_performance"
EMBEDDING_ENDPOINT: "databricks-gte-large-en"
EMBEDDING_DIMS: 1024
APP_NAME: "{app_name}"
"""

with open(config_file_path, "w") as f:
    f.write(config_content)

In [ ]:
# Configuration Values summary table (links + copy buttons)
from Includes import display_config_values

display_config_values([
    ("Catalog",          f'<a href="/explore/data/{catalog_name}" target="_blank">{catalog_name}</a>', True),
    ("Schema",           f'<a href="/explore/data/{catalog_name}/{schema_name}" target="_blank">{catalog_name}.{schema_name}</a>', f"{catalog_name}.{schema_name}"),
    ("Lakebase project", lakebase_project_name, True),
    ("Lakebase branch",  "production"),
    ("App name",         app_name, True),
    ("App folder",       "bakehouse-agent-app/"),
    ("Experiment",       f'<a href="{experiment_location}" target="_blank">{experiment_location}</a>'),
    ("Config file",      config_file_path),
])